# Downloading JWST MIRI MRS data from MAST

**Target:** SY Cha (a Class II protoplanetary disk in Chamaeleon, ~150 pc) — this is the default example object for the rest of the workshop.

**Goal of this notebook:** learn how to *discover* what JWST data exists for a given target and how to *download the raw (uncalibrated) files* programmatically, so you can re-run the calibration pipeline yourself in [`MRSReduction/`](../MRSReduction/JWPipeNB-MIRI-MRS.ipynb).

We do this in two complementary ways:

1. **Mission-specific search** with `astroquery.mast.MastMissions` — the modern MAST API, optimised for JWST/HST-specific metadata fields.

2. **Cross-mission search** with `astroquery.mast.Observations` — the older but more flexible interface, which is how almost all existing tutorials and scripts in the wild are written.

Both end with the same step: select the raw `_uncal.fits` products and download them with `Observations.download_products`.

> The web equivalent of everything below is the [MAST Portal](https://mast.stsci.edu/portal/Mashup/Clients/Mast/Portal.html). It is great for *browsing* — but for reproducible bulk downloads you want a script you can re-run.

## Prerequisites

We need the **pre-release** of `astroquery`. The dev version adds two JWST-specific features we rely on below:

- a `flat=True` option for `download_products`, which puts everything in a single directory instead of MAST's nested `mastDownload/JWST/...` tree;

- de-duplication of already-downloaded files, so re-running the notebook is cheap.

If you do not have it yet, install it once into your `mrs_analysis` environment:

```bash
mamba activate mrs_analysis
pip install --pre --upgrade astroquery
```

Public JWST data (older than 12 months, or non-proprietary) does **not** require a MAST account. SY Cha's MIRI MRS data has been public since 2023, so no login is needed for this notebook. If you ever want to download proprietary data, set the environment variable `MAST_API_TOKEN` (see the [MyST auth page](https://auth.mast.stsci.edu/info)).

In [ ]:
from pathlib import Path
import numpy as np
import astroquery
from astropy.table import vstack, unique
from astroquery.mast import MastMissions, Observations

print(f'astroquery version: {astroquery.__version__}')
# We need >= 0.4.10 for flat=True. The pre-release advertises itself with a
# trailing 'dev' suffix. Older versions still work, you just lose flat=True.

## Part 1 — Mission-specific search (`MastMissions`)

`MastMissions` is the newer interface; it exposes mission-specific columns (here, all the JWST-pipeline keywords like `instrume`, `filter`, `exp_type`, `opticalElements`, etc.) directly. Underlying reference: the [STScI mission-search notebook](https://spacetelescope.github.io/mast_notebooks/notebooks/multi_mission/missions_mast_search/missions_mast_search.html).

Here we ask: *what JWST observations exist for SY Cha?* Then we narrow to the MIRI MRS exposure type (`MIR_MRS`).

In [ ]:
missions = MastMissions(mission='jwst')

# List of available column names — uncomment if you want to see them all.
# print(missions.get_column_list()[['name', 'description']])

# Resolve the target name and pull every JWST observation within 30".
# select_cols=[...] explicitly requests the header TARGNAME column ('targname'),
# which is NOT in the default column set returned by query_object.
jwst_all = missions.query_object(
    'SY Cha',
    radius=30,  # arcsec
    select_cols=[
        'fileSetName', 'targname', 'targprop',
        'instrume', 'exp_type', 'opticalElements',
        'productLevel',
    ],
    # Tip: `missions.get_column_list()` shows every column the JWST schema
    # exposes (there are 200+), including various date / exposure-time fields.
)
print(f'Found {len(jwst_all)} JWST datasets within 30" of SY Cha.')

In [ ]:
# Display a useful subset of columns. Re-run this cell freely — it does not
# re-query MAST. `targname` is the FITS TARGNAME keyword (the canonical name
# resolved by APT at proposal time); `targprop` is the proposer's free-form
# APT label. Showing both makes it obvious when the cone search has picked
# up an unrelated nearby source.
jwst_all[['fileSetName', 'targname', 'targprop', 'instrume', 'exp_type', 'opticalElements', 'productLevel']][:10]

In [ ]:
# Narrow to MIRI MRS exposures only. We filter the cone-search table
# CLIENT-SIDE instead of issuing a new query_criteria(targname='SY Cha', ...)
# call — different programs record different canonical names in the TARGNAME
# header keyword (SY-CHA, [NC98] Cha HA 1, Sz 1, ...), so an exact string
# match on targname is fragile. The cone search already restricted us to
# the right patch of sky, so we just keep the MRS exposures from there.
# exp_type = MIR_MRS  ->  MIRI Medium-Resolution Spectroscopy (the IFU mode).
miri_mrs = jwst_all[jwst_all['exp_type'] == 'MIR_MRS']
print(f'{len(miri_mrs)} MIRI MRS exposures within 30" of SY Cha.')

In [ ]:
miri_mrs[['fileSetName', 'targname', 'targprop', 'exp_type', 'opticalElements', 'productLevel']]

The `fileSetName` column is the per-exposure root (e.g. `jw01282030001_03101_00001_mirifulong`) — the same root that prefixes every calibration-stage file in MAST. `productLevel` tells you which calibration levels are available (1, 2, 3, with 1 = raw `_uncal.fits`).

**About the target-name columns:** `targname` is the `TARGNAME` FITS keyword (the canonical name resolved by APT at proposal time); `targprop` is the proposer's free-form label (the name they typed into APT). They usually agree, but `targprop` can carry program-specific tags like `'SY-Cha-MIRI'` or `'MINDS-SYCha'`. Showing both makes it obvious when the cone search has picked up an unrelated nearby source.

Note the proposal ID embedded in `fileSetName`: **1282** — this is the MINDS GTO program (Kamp+ / Henning+; PI Henning). Knowing the proposal ID makes downstream filtering much easier.

## Part 2 — Cross-mission search (`Observations`)

The older `Observations` interface treats *all* MAST missions (HST, JWST, Kepler, TESS, …) the same way — same column names, same query syntax. Most existing tutorials and scripts use it, so it's worth knowing.

Two query styles are useful:

- `query_criteria(target_name=..., instrument_name=...)` — fastest, but `target_name` is the **exact string** stored in the FITS header (so `'SY Cha'` matches but `'sy cha'` does not).

- `query_criteria(objectname=..., radius=...)` — resolves the name via Simbad/NED, then does a *cone search*. More robust when the header target name is non-standard.

In [ ]:
# 'MIRI/IFU' is the MAST instrument_name for MIRI MRS
# (the same instrument exposes 'MIRI/IMAGE' for imaging and 'MIRI/CORON' for coron).
obs = Observations.query_criteria(
    objectname='SY Cha',
    radius='30s',
    obs_collection='JWST',
    instrument_name='MIRI/IFU',
)
print(f'{len(obs)} JWST MIRI/IFU observations near SY Cha.')

In [ ]:
obs['proposal_id', 'obs_id', 'target_name', 'filters', 't_min', 't_exptime', 'dataproduct_type']

**Important difference between the two APIs.** `MastMissions` (Part 1) returns one row per *exposure* — every single `_uncal.fits` / `_rate.fits` / `_cal.fits` / … file is visible in the table, and `productLevel` tells you which calibration stage each one is at. `Observations.query_criteria` (this section) is coarser: it returns one row per *MAST observation*, which for JWST is the **Stage-3 combined dataset** for each (channel, band) — i.e. only the top-level Level-3 cubes and 1D spectra show up here. Notice that `calib_level` is `3` for every row above.

The raw and intermediate files (Levels 1–2c) are *not* missing — they are just one step further down. We surface them in **Part 3** with `Observations.get_product_list(obs)`, which expands each top-level observation into every underlying file at every calibration level.

An *observation* in MAST's sense for the MRS usually corresponds to one (channel, band) combination, not one exposure — so 12 entries for one science target is normal (3 channels × 4 sub-bands).

From here on we stick to a single program. Filter by `proposal_id='1282'` to keep only the MINDS GTO data.

In [ ]:
minds = obs[obs['proposal_id'] == '1282']
print(f'MINDS (proposal 1282) MIRI/IFU observations of SY Cha: {len(minds)}')

In [ ]:
minds['proposal_id', 'obs_id', 'filters', 't_exptime', 'calib_level']

## Part 3 — From observations to *products*

An observation is a logical grouping; a **product** is an actual file on disk. `get_product_list` returns every file MAST has for each observation — at every calibration level.

Calibration levels for JWST. The headline number (1 / 2 / 3) follows the [CAOM standard](https://www.opencadc.org/caom2/) used by every MAST mission; the letter suffix (`1b`, `2a`, `2b`, `2c`) is JWST-specific and maps one-to-one onto the pipeline stages:

| Level | Pipeline stage | Suffix(es) | What it is |
|:-----:|:---------------|:-----------|:-----------|
| **1b** | none (raw) | `_uncal.fits` | Raw, uncalibrated detector ramps straight off the spacecraft. *This is what we download below.* |
| **2a** | Stage 1 (`Detector1`) | `_rate.fits`, `_rateints.fits` | Slope images from ramp-fit — bias / dark / linearity / jump corrections applied, but no flux calibration yet. |
| **2b** | Stage 2 (`Spec2` / `Image2`) | `_cal.fits`, `_calints.fits`, `_x1d.fits`, `_s2d.fits` | Per-exposure photometrically and spectroscopically calibrated 2D images / 1D spectra. For MRS, each `_cal.fits` is one IFU sub-band of one dither. |
| **2c** | Stage 2 special | varies (`_crf.fits`, AMI/Coron products) | Stage-2 outputs for modes that need extra per-exposure processing (TSO light curves, AMI interferograms, coronagraphic PSF-subtracted frames). Empty for vanilla MRS. |
| **3** | Stage 3 (`Spec3` / `Image3` / `Cube3`) | `_s3d.fits`, `_x1d.fits`, `_i2d.fits`, `_asn.json` | Ensemble products combining all exposures of a target: MRS cubes and combined 1D spectra, imaging mosaics, level-3 association files. |

If you only want the final science-ready cubes and spectra, grab level 3 and skip the pipeline. If you want to reprocess (e.g. with a newer pipeline version, custom rejection, or non-default cube-building parameters — which we do in this workshop), you want level 1b: `_uncal.fits`.

We query each observation in a loop and `vstack` the results — this is faster and more reliable than the server-side join you would get from passing the whole table at once.

In [ ]:
product_lists = [Observations.get_product_list(o) for o in minds]
products = vstack(product_lists)
print(f'{len(products)} products across {len(minds)} observations.')

In [ ]:
# How are they distributed across calibration levels?
# productSubGroupDescription is a MaskedColumn — many rows (e.g. _asn.json
# association files) have no sub-group, so we replace masked entries with
# a sentinel string before counting (MaskedConstant is unhashable).
import collections
levels = products['calib_level'].tolist()
subs = products['productSubGroupDescription'].filled('(none)').tolist()
counts = collections.Counter(zip(levels, subs))
for (lvl, sub), n in sorted(counts.items()):
    print(f'  level {lvl}  {sub:<14}  {n:4d} files')

## Part 4 — Selecting just the raw `_uncal.fits` files

For this workshop we want the **raw, uncalibrated science exposures** so we can re-run the full pipeline. That means:

- `_uncal.fits` filename suffix (level 1),

- **excluding guide-star products** (`_gs-*` files — astrometry frames from the FGS, ~10× more numerous than the science files and never used for science).

In [ ]:
# `dataURI` is an astropy MaskedColumn with a string fill-value ('N/A'),
# which trips up np.char.find / endswith when chained with comparisons.
# Convert to a plain Python list of strings and do the matching in pure
# Python — clearer to read, and immune to the masked-array gotcha.
uris = products['dataURI'].tolist()
is_uncal = np.array([u.endswith('_uncal.fits') for u in uris])
is_guidestar = np.array(['_gs-' in u for u in uris])

# Keep only the MRS detector exposures. The MIRI detector tag in the
# filename tells you which optics path the photons travelled:
#   mirifushort  -> MRS short-wavelength detector (channels 1 & 2)
#   mirifulong   -> MRS long-wavelength detector  (channels 3 & 4)
#   mirimage     -> MIRI imager — used here for target acquisition only
# `mirifu` matches both MRS detectors and excludes the TA image. Drop
# this mask (or change it to `'miri' in u`) if you also want the TA frame.
is_mrs = np.array(['mirifu' in u for u in uris])

raw = products[is_uncal & ~is_guidestar & is_mrs]
print(f'{len(raw)} raw _uncal rows before de-duplication.')

# DE-DUPLICATE. Each of the 12 Level-3 observations in `minds` (one per
# channel/band) references the SAME underlying _uncal.fits exposures, so
# after vstack every raw file appears ~12 times. Without this step we
# would queue ~300 downloads for ~25 unique files — and tell MAST so.
raw = unique(raw, keys='dataURI')
print(f'{len(raw)} unique raw _uncal.fits files after de-duplication.')

# Quick sanity check: total download size in GB.
size_gb = raw['size'].sum() / 1024**3
print(f'Total size on disk: {size_gb:.2f} GB')
raw['obsID', 'productFilename', 'size', 'calib_level'][:8]

## Part 5 — Download

We download into [`../data/SYCha/uncal/`](../data/SYCha/) so the files land exactly where the pipeline notebook in [`MRSReduction/`](../MRSReduction/) expects them. `flat=True` flattens MAST's nested directory layout into one directory; if you skip it you get `mastDownload/JWST/<obs_id>/...` instead.

Re-running this cell is safe — already-downloaded files are detected and skipped.

In [ ]:
download_dir = Path('../data/SYCha/uncal').resolve()
download_dir.mkdir(parents=True, exist_ok=True)
print(f'Downloading to: {download_dir}')

manifest = Observations.download_products(
    raw,
    download_dir=str(download_dir),
    flat=True,
)
manifest

## Verification & next steps

Spot-check the directory: you should see one `_uncal.fits` file per exposure, all sitting in the same flat directory.

In [ ]:
uncal_files = sorted(download_dir.glob('*_uncal.fits'))
print(f'{len(uncal_files)} _uncal.fits files in {download_dir}')
for f in uncal_files[:5]:
    print(' ', f.name)
if len(uncal_files) > 5:
    print(f'  ... and {len(uncal_files) - 5} more')

**Where to go from here:**

- Open [`../MRSReduction/JWPipeNB-MIRI-MRS.ipynb`](../MRSReduction/JWPipeNB-MIRI-MRS.ipynb) and point it at this `uncal/` directory — Stages 1–3 will calibrate the raw data into the `_s3d.fits` cubes and `_x1d.fits` spectra you already see in [`../data/SYCha/stage3/`](../data/SYCha/stage3/).

- For other targets, change `objectname='SY Cha'` to any Simbad-resolvable name. For other instruments, use e.g. `instrument_name='NIRSPEC/IFU'`, `'NIRCAM/IMAGE'`, `'MIRI/IMAGE'`. The full list of MAST instrument names is in the [astroquery.mast docs](https://astroquery.readthedocs.io/en/latest/mast/mast.html).

- For a more JWST-aware bulk-download workflow (filter by detector, by obsnum, etc.), use STScI's [`jwst_mast_query`](https://github.com/spacetelescope/jwst_mast_query) — there is an example config in [`../data/SYCha/jwst_query_SYCha.cfg`](../data/SYCha/jwst_query_SYCha.cfg).